# T4 — Format Comparison: Green Taxi 2024
Sara Milovanova & Biljana Vitanova

Source: `partitioned/green_tripdata/2024/part-0.parquet` (~18 MB, ~660k rows)

The same dataset is exported to four formats and compared on:
- **File size on disk**
- **Time to read into a Pandas DataFrame** (median of 5 runs)

| Format | Description |
|---|---|
| Parquet | Columnar, compressed binary — baseline |
| CSV | Plain text, uncompressed |
| CSV.gz | Plain text, gzip-compressed |
| HDF5 | Hierarchical binary, optimised for row access |
| DuckDB | Embedded analytical database file |

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from pathlib import Path

T4 = Path("/d/hpc/projects/FRI/bigdata/students/sm_bv/final_project/data/t4")
df = pd.read_csv(T4 / "benchmark.csv")
df = df.set_index("format")
print(f"Dataset: {df['rows'].iloc[0]:,} rows × {df['columns'].iloc[0]} columns")
df[["size_mb", "read_time_s"]]

---
## 1. File Size Comparison

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

PLOTS = Path("/d/hpc/projects/FRI/bigdata/students/sm_bv/final_project/data/t4/plots")
PLOTS.mkdir(exist_ok=True)

size_colors = ["#003f5c","#2f6690","#2196a0","#3dac88","#7bc67e"]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(df.index, df["size_mb"], color=size_colors, edgecolor="black", linewidth=0.5)
for bar, val in zip(bars, df["size_mb"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{val:.1f} MB", ha="center", va="bottom", fontsize=13)
ax.set_title("File Size by Format — Green Taxi 2024", fontsize=16)
ax.set_ylabel("Size (MB)", fontsize=14)
ax.set_xlabel("Format", fontsize=14)
ax.tick_params(axis="both", labelsize=13)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS / "t4_file_sizes.pdf", bbox_inches="tight")
plt.show()

---
## 2. Read Speed Comparison

In [ ]:
time_colors = ["#ff6b35","#f7c59f","#efefd0","#004e89","#1a936f"]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(df.index, df["read_time_s"], color=time_colors, edgecolor="black", linewidth=0.5)
for bar, val in zip(bars, df["read_time_s"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f"{val:.3f}s", ha="center", va="bottom", fontsize=13)
ax.set_title("Read Time into Pandas DataFrame (median of 5 runs)", fontsize=16)
ax.set_ylabel("Seconds", fontsize=14)
ax.set_xlabel("Format", fontsize=14)
ax.tick_params(axis="both", labelsize=13)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS / "t4_read_times.pdf", bbox_inches="tight")
plt.show()

---
## 3. Size vs Speed Trade-off

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for i, (fmt, row) in enumerate(df.iterrows()):
    ax.scatter(row["size_mb"], row["read_time_s"], s=120, color=colors[i], zorder=3)
    ax.annotate(fmt, (row["size_mb"], row["read_time_s"]),
                textcoords="offset points", xytext=(6, 4), fontsize=10)
ax.set_xlabel("File size (MB)")
ax.set_ylabel("Read time (s)")
ax.set_title("Size vs Read Speed Trade-off", fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 4. Summary Table

In [ ]:
summary = df[["size_mb","read_time_s"]].copy()
summary["size_vs_parquet"]    = (summary["size_mb"]    / summary.loc["parquet","size_mb"]).round(2)
summary["speed_vs_parquet"]   = (summary["read_time_s"] / summary.loc["parquet","read_time_s"]).round(2)
summary.columns = ["Size (MB)","Read time (s)","Size ratio vs parquet","Time ratio vs parquet"]
summary

---
## 5. Conclusions

| Format | Best for | Drawback |
|---|---|---|
| **Parquet** | Analytics, storage efficiency, skip-reading individual columns | Binary — not human-readable, requires a parquet-aware tool |
| **CSV** | Human-readable, universally compatible (Excel, any language) | Largest file on disk, slowest to parse |
| **CSV.gz** | Reducing storage when CSV compatibility is required | Slow read — must decompress before parsing |
| **HDF5** | Fast full-table reads, appending rows incrementally | Larger file than parquet, requires the `tables` package, designed for scientific arrays not tabular data exchange |
| **DuckDB** | Running SQL queries directly on the stored data | File is only readable with DuckDB — not portable to other tools |